In [1]:
print("hello")

hello


In [2]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("../data/raw/dataset.csv")

print(df.shape)

(975, 30)


In [3]:
def get_strike(col):
    return int(
        re.search(
            r'JAN26(\d+)(CE|PE)$',
            col
        ).group(1)
    )

def get_option_type(col):
    return col[-2:]

In [4]:
records = []

option_cols = [
    c for c in df.columns
    if c.endswith("CE") or c.endswith("PE")
]

for row_idx, row in df.iterrows():

    spot = row["underlying_price"]

    for col in option_cols:

        records.append({
            "row_idx": row_idx,
            "datetime": row["datetime"],
            "underlying_price": spot,
            "strike": get_strike(col),
            "option_type": get_option_type(col),
            "iv": row[col]
        })

long_df = pd.DataFrame(records)

print(long_df.shape)
long_df.head()

(27300, 6)


,row_idx,datetime,underlying_price,strike,option_type,iv
0,0,07-01-2026 09:15,26111.65,25200,CE,0.12662
1,0,07-01-2026 09:15,26111.65,25300,CE,0.12330
2,0,07-01-2026 09:15,26111.65,25400,CE,0.11741
3,0,07-01-2026 09:15,26111.65,25500,CE,NaN
4,0,07-01-2026 09:15,26111.65,25600,CE,0.11005


In [5]:
long_df["datetime"] = pd.to_datetime(
    long_df["datetime"],
    dayfirst=True
)

long_df["hour"] = long_df["datetime"].dt.hour
long_df["minute"] = long_df["datetime"].dt.minute

long_df["time_index"] = (
    long_df["datetime"]
    - long_df["datetime"].min()
).dt.total_seconds() / 60

long_df[
    ["datetime","hour","minute","time_index"]
].head()

,datetime,hour,minute,time_index
0,2026-01-07 09:15:00,9,15,0.0
1,2026-01-07 09:15:00,9,15,0.0
2,2026-01-07 09:15:00,9,15,0.0
3,2026-01-07 09:15:00,9,15,0.0
4,2026-01-07 09:15:00,9,15,0.0


In [6]:
long_df["moneyness"] = (
    long_df["strike"]
    / long_df["underlying_price"]
)

long_df["distance_from_atm"] = (
    long_df["strike"]
    - long_df["underlying_price"]
)

long_df[
    [
        "underlying_price",
        "strike",
        "moneyness",
        "distance_from_atm"
    ]
].head()

,underlying_price,strike,moneyness,distance_from_atm
0,26111.65,25200,0.965086,-911.65
1,26111.65,25300,0.968916,-811.65
2,26111.65,25400,0.972746,-711.65
3,26111.65,25500,0.976576,-611.65
4,26111.65,25600,0.980405,-511.65


In [7]:
long_df["option_type"] = (
    long_df["option_type"]
    .map({
        "CE":0,
        "PE":1
    })
)

long_df["option_type"].value_counts()

option_type
0    13650
1    13650
Name: count, dtype: int64

In [8]:
train_df = long_df[
    long_df["iv"].notna()
].copy()

print(train_df.shape)

(21840, 11)


In [9]:
known_idx = train_df.index

np.random.seed(42)

val_idx = np.random.choice(
    known_idx,
    size=int(
        0.2 * len(known_idx)
    ),
    replace=False
)

train_idx = np.setdiff1d(
    known_idx,
    val_idx
)

print("Train:", len(train_idx))
print("Validation:", len(val_idx))

Train: 17472
Validation: 4368


In [10]:
features = [
    "underlying_price",
    "strike",
    "moneyness",
    "distance_from_atm",
    "option_type",
    "hour",
    "minute",
    "time_index"
]

features

['underlying_price',
 'strike',
 'moneyness',
 'distance_from_atm',
 'option_type',
 'hour',
 'minute',
 'time_index']

In [11]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error

In [12]:
model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    random_state=42
)

model.fit(
    train_df.loc[
        train_idx,
        features
    ],
    train_df.loc[
        train_idx,
        "iv"
    ]
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000385 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1070
[LightGBM] [Info] Number of data points in the train set: 17472, number of used features: 8
[LightGBM] [Info] Start training from score 0.182868


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.03
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [13]:
preds = model.predict(
    train_df.loc[
        val_idx,
        features
    ]
)

mse = mean_squared_error(
    train_df.loc[
        val_idx,
        "iv"
    ],
    preds
)

print("LightGBM Validation MSE:", mse)

LightGBM Validation MSE: 0.0012827852501626823
